In [1]:
import os
import pandas as pd
import chromadb
from openai import OpenAI

DATA_PATH = "data/notes.csv"
CHROMA_PATH = "chroma_store"
COLLECTION_NAME = "study_notes"

client = OpenAI(
    base_url="https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1",
    api_key="any value",
    default_headers={"x-api-key": os.getenv("API_GATEWAY_KEY")}
)

In [2]:
%load_ext dotenv
%dotenv ../../05_src/.secrets

In [3]:
pd.read_csv(DATA_PATH).head()

,id,title,text,source
0,1,Learning Python,Python is a beginner-friendly programming lang...,study_notes
1,2,Prompt Engineering,Prompt engineering is the practice of designin...,study_notes
2,3,Embeddings,Embeddings convert text into numerical vectors...,study_notes
3,4,Vector Databases,Vector databases store embeddings and allow ef...,study_notes
4,5,RAG,Retrieval-augmented generation combines docume...,study_notes


In [4]:

def get_embedding(text: str):
    response = client.embeddings.create(
        model="text-embedding-3-small",
        input=text
    )
    return response.data[0].embedding

def main():
    df = pd.read_csv(DATA_PATH)

    chroma_client = chromadb.PersistentClient(path=CHROMA_PATH)
    collection = chroma_client.get_or_create_collection(name=COLLECTION_NAME)

    for _, row in df.iterrows():
        doc_id = str(row["id"])
        title = str(row["title"])
        text = str(row["text"])
        source = str(row["source"])

        embedding = get_embedding(text)

        collection.upsert(
            ids=[doc_id],
            documents=[text],
            embeddings=[embedding],
            metadatas=[{"title": title, "source": source}]
        )

    print(f"Indexed {len(df)} notes into ChromaDB at {CHROMA_PATH}")

In [6]:
main()

Indexed 12 notes into ChromaDB at chroma_store
